In [149]:
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pyarrow as pa


@dataclass(frozen=True)
class ValidationResult:
    status: str       # 'OK' | 'EMPTY' | 'TICKER_REUSE' | 'PARTIAL_HISTORY'
    reason: str       # human-readable detail
    yf_start: pd.Timestamp | None
    yf_end: pd.Timestamp | None
    n_rows: int

ticker = 'AAPL'
spell_start = pd.Timestamp('2020-1-1')
spell_end = pd.NaT
result = ValidationResult(
    status='OK',
    reason='GOOD!',
    yf_start=spell_start,
    yf_end=spell_end,
    n_rows=1
)

data = {
    'ticker': [ticker],
    'attempted_at': [pd.Timestamp.now()],
    'spell_start': [spell_start],
    'spell_end': [spell_end],
    'yf_start': [result.yf_start],
    'yf_end': [result.yf_end],
    'status': [result.status],
    'reason': [result.reason],
    'n_rows': [result.n_rows]
}

# ledger = pd.DataFrame(data)

row_dict = {
    'ticker': ticker,
    'attempted_at': pd.Timestamp.now(),
    'spell_start': spell_start,
    'spell_end': spell_end,
    'yf_start': result.yf_start,
    'yf_end': result.yf_end,
    'status': result.status,
    'reason': 'BINGUS',
    'n_rows': result.n_rows
    }   

home_path = Path('/Users/eggs')
ledger_file_path = home_path / '_ledger.parquet'

try:
    ledger = pd.read_parquet(ledger_file_path)
    ledger = pd.concat([ledger, pd.DataFrame([row_dict])])
    ledger = ledger.sort_values(by=['attempted_at'], ascending=True)
    ledger.to_parquet(ledger_file_path, index=True)
except (FileNotFoundError, OSError, ValueError, pa.lib.ArrowException) as e:
    print(f"Ledger file does not exist or appears to be corrupted at {ledger_file_path} ({e!r}); creating new ledger.")
    ledger = pd.DataFrame(row_dict, index=[0])
    ledger.to_parquet(ledger_file_path, index=True)
    print(f"Ledger saved to {ledger_file_path}.")

print(ledger)

  ticker               attempted_at spell_start spell_end   yf_start yf_end  \
0   AAPL 2026-06-14 23:46:13.118389  2020-01-01       NaT 2020-01-01    NaT   
0   AAPL 2026-06-14 23:46:41.055813  2020-01-01       NaT 2020-01-01    NaT   
0   AAPL 2026-06-14 23:46:46.273936  2020-01-01       NaT 2020-01-01    NaT   

  status                                reason  n_rows  
0     OK  Price history appears to be correct.       1  
0     OK                                BINGUS       1  
0     OK                                BINGUS       1  


In [122]:
df1 = pd.DataFrame({'A': 1, 'B': 2})
df2 = pd.DataFrame({'A':3, 'B':4})

df1

ValueError: If using all scalar values, you must pass an index